# TrainLM on TPU

This is the same Hugging Face-like workflow an end user runs. Choose a TPU runtime, fill in the four values below, and call `trainer.train()`.

TrainLM handles TPU discovery, world size, worker launch, ranks, preflight, caching, distributed data ownership, checkpoints, evaluation, and structured results under the hood.

## 1. Install

Run from a checked-out TrainLM repository, then restart the notebook kernel.

In [ ]:
%pip install -e ".[tpu-xla]" -c constraints/tpu-xla-2.9.txt

## 2. Your inputs

Use an immutable 40-character Hugging Face commit SHA for a reproducible TPU run. Each data directory contains packed-bin manifests and their referenced shard files.

In [ ]:
from pathlib import Path

MODEL_ID = "microsoft/Phi-3.5-mini-instruct"
MODEL_REVISION = "PUT_THE_40_CHARACTER_HF_COMMIT_SHA_HERE"
TRAIN_DATA = Path("/kaggle/input/trainlm-packed/train")
EVAL_DATA = Path("/kaggle/input/trainlm-packed/eval")
OUTPUT_DIR = Path("/kaggle/working/trainlm-run")

## 3. Build the datasets and trainer

Dataset construction validates the manifests and payloads before training. There is no TPU topology configuration in this notebook.

In [ ]:
from trainlm import PackedBinDataset, TrainLMTrainer, TrainLMTrainingArguments

sequence_length = 2048
train_dataset = PackedBinDataset.from_directory(
    TRAIN_DATA,
    sequence_length=sequence_length,
    split="train",
)
eval_dataset = PackedBinDataset.from_directory(
    EVAL_DATA,
    sequence_length=sequence_length,
    split="validation",
)

trainer = TrainLMTrainer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=TrainLMTrainingArguments(
        output_dir=OUTPUT_DIR,
        accelerator="tpu",
        bf16=True,
        max_steps=6,
        sequence_length=sequence_length,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        logging_steps=1,
        eval_steps=2,
        save_steps=2,
    ),
)

## 4. Train

This single call performs the private collective probe and model preflight, launches every available TPU worker, trains, evaluates every two steps, and writes committed checkpoints every two steps.

In [ ]:
result = trainer.train()
result

## 5. Optional: inspect what TrainLM selected

`explain()` is useful when reviewing fallbacks or filing a result. It does not require users to inspect worker commands or logs.

In [ ]:
trainer.explain(format="text")

## 6. Optional: resume

Normally set this to the last committed checkpoint after an interrupted or completed run. TrainLM validates topology and restores model, optimizer, scheduler, runtime, RNG, trainer, and packed-data position internally.

In [ ]:
resumed_trainer = TrainLMTrainer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=TrainLMTrainingArguments(
        output_dir=Path("/kaggle/working/trainlm-resumed"),
        accelerator="tpu",
        bf16=True,
        max_steps=10,
        sequence_length=sequence_length,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        logging_steps=1,
        eval_steps=2,
        save_steps=2,
    ),
)
resumed_result = resumed_trainer.train(
    resume_from_checkpoint=OUTPUT_DIR / "checkpoint-4"
)
resumed_result

## What to save from a validation run

Archive the output directory, including `coordinator_summary.json`, `summary.json`, `metrics.jsonl`, committed checkpoint manifests/shards, and XLA metrics. The returned result is the normal user-facing status; these files are only needed for debugging or performance certification.

A successful run validates the lifecycle on that TPU. It does not by itself mark performance as certified.